In [4]:
import warnings
warnings.simplefilter(action='ignore')

import os
import datetime, pandas

from tqdm import tqdm
from dataclasses import dataclass, asdict

import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Union
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import TimeSeriesSplit
import warnings
from abc import ABC, abstractmethod

import kaggle_evaluation.default_inference_server

from catboost import CatBoostRegressor, Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoLars
from sklearn.linear_model import Lasso
from sklearn.linear_model import BayesianRidge
from sklearn.linear_model import TweedieRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression

from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestCentroid

from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.naive_bayes import CategoricalNB

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import StackingRegressor

from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

In [17]:
from pathlib import Path
from dataclasses import dataclass
import polars as pl
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from typing import List, Optional, Union, Tuple

@dataclass
class PreprocessedData:
    """Container for preprocessed data"""
    data: pl.DataFrame
    features: List[str]
    scaler: Optional[StandardScaler] = None
    target_column: Optional[str] = None

@dataclass 
class ElasticNetParameters:
    l1_ratio: float 
    cv: int
    alphas: np.ndarray 
    max_iter: int 
    
    def __post_init__(self): 
        if self.l1_ratio < 0 or self.l1_ratio > 1: 
            raise ValueError("Wrong initializing value for ElasticNet l1_ratio")
        
@dataclass(frozen=True)
class RetToSignalParameters:
    signal_multiplier = 400.0
    min_signal = 0.0
    max_signal = 2.0

def preprocess_hull_tactical_data(
    df: Union[pl.DataFrame, pd.DataFrame],
    vars_to_keep: Optional[List[str]] = None,
    target_column: Optional[str] = None,
    id_column: str = 'date_id',
    scale_features: bool = True,
    create_engineered_features: bool = True
) -> PreprocessedData:
    """
    Preprocess Hull Tactical Market Prediction DataFrame with feature engineering and scaling.
    
    This function handles:
    1. Feature engineering (creating U1 and U2 features)
    2. Feature selection
    3. Data cleaning and null handling
    4. Optional feature scaling
    5. Returns preprocessed data ready for modeling
    
    Args:
        df (Union[pl.DataFrame, pd.DataFrame]): Input dataframe to preprocess
        vars_to_keep (Optional[List[str]]): List of variables to keep in final dataset.
                                          If None, uses default feature set.
        target_column (Optional[str]): Name of target column. If None, no target processing.
        id_column (str): Name of ID column to preserve (default: 'date_id')
        scale_features (bool): Whether to apply StandardScaler to features
        create_engineered_features (bool): Whether to create U1 and U2 engineered features
    
    Returns:
        PreprocessedData: Container with preprocessed dataframe, feature list, and scaler
        
    Example:
        >>> # Basic preprocessing
        >>> processed = preprocess_hull_tactical_data(df)
        >>> 
        >>> # With target column
        >>> processed = preprocess_hull_tactical_data(df, target_column='target')
        >>> 
        >>> # Custom feature set
        >>> custom_features = ['S1', 'S2', 'E2', 'P9']
        >>> processed = preprocess_hull_tactical_data(df, vars_to_keep=custom_features)
    """
    
    # Convert to Polars if pandas DataFrame
    if isinstance(df, pd.DataFrame):
        df = pl.from_pandas(df)
    elif not isinstance(df, pl.DataFrame):
        raise ValueError("Input must be a pandas or polars DataFrame")
    
    # Default feature set
    if vars_to_keep is None:
        vars_to_keep = [
            "S2", "E2", "E3", "P9", "S1", "S5", "I2", "P8",
            "P10", "P12", "P13"
        ]
        if create_engineered_features:
            vars_to_keep.extend(["U1", "U2"])
    
    print(f"Input data shape: {df.shape}")
    
    # Step 1: Cast numeric columns (excluding ID column)
    print("Casting numeric columns...")
    df_processed = df.with_columns(
        pl.exclude(id_column).cast(pl.Float64, strict=False)
    )
    
    # Step 2: Create engineered features if requested
    if create_engineered_features:
        print("Creating engineered features...")
        required_cols = ["I1", "I2", "I7", "I9", "M11"]
        missing_cols = [col for col in required_cols if col not in df_processed.columns]
        
        if missing_cols:
            print(f"Warning: Cannot create engineered features. Missing columns: {missing_cols}")
            # Remove U1, U2 from vars_to_keep if they can't be created
            vars_to_keep = [col for col in vars_to_keep if col not in ["U1", "U2"]]
        else:
            df_processed = df_processed.with_columns([
                (pl.col("I2") - pl.col("I1")).alias("U1"),
                (pl.col("M11") / ((pl.col("I2") + pl.col("I9") + pl.col("I7")) / 3)).alias("U2")
            ])
    
    # Step 3: Select columns to keep
    columns_to_select = [id_column]
    if target_column and target_column in df_processed.columns:
        columns_to_select.append(target_column)
    
    # Filter vars_to_keep to only include columns that exist
    available_features = [col for col in vars_to_keep if col in df_processed.columns]
    missing_features = [col for col in vars_to_keep if col not in df_processed.columns]
    
    if missing_features:
        print(f"Warning: The following requested features are not available: {missing_features}")
    
    columns_to_select.extend(available_features)
    
    print(f"Selecting columns: {columns_to_select}")
    df_processed = df_processed.select(columns_to_select)
    
    # Step 4: Handle null values
    print("Handling null values...")
    df_processed = df_processed.with_columns([
        pl.col(col).fill_null(pl.col(col).ewm_mean(com=0.5))
        for col in available_features
    ]).drop_nulls()
    
    print(f"Data shape after cleaning: {df_processed.shape}")
    
    # Step 5: Feature scaling if requested
    scaler = None
    if scale_features and available_features:
        print("Scaling features...")
        scaler = StandardScaler()
        
        # Extract feature data for scaling
        feature_data = df_processed.select(available_features)
        scaled_data = scaler.fit_transform(feature_data.to_numpy())
        
        # Create scaled feature DataFrame
        scaled_features_df = pl.from_numpy(scaled_data, schema=available_features)
        
        # Combine with non-feature columns
        non_feature_cols = [col for col in df_processed.columns if col not in available_features]
        if non_feature_cols:
            non_feature_data = df_processed.select(non_feature_cols)
            df_processed = pl.concat([non_feature_data, scaled_features_df], how="horizontal")
        else:
            df_processed = scaled_features_df
    
    print("Preprocessing completed successfully!")
    print(f"Final shape: {df_processed.shape}")
    print(f"Features: {available_features}")
    
    return PreprocessedData(
        data=df_processed,
        features=available_features,
        scaler=scaler,
        target_column=target_column
    )

def split_preprocessed_data(
    processed_data: PreprocessedData,
    test_condition: Optional[pl.Expr] = None,
    train_condition: Optional[pl.Expr] = None,
    test_size: Optional[float] = None,
    id_column: str = 'date_id'
) -> Tuple[pl.DataFrame, pl.DataFrame, Optional[pl.Series], Optional[pl.Series]]:
    """
    Split preprocessed data into train and test sets.
    
    Args:
        processed_data (PreprocessedData): Output from preprocess_hull_tactical_data
        test_condition (Optional[pl.Expr]): Polars expression to filter test data
        train_condition (Optional[pl.Expr]): Polars expression to filter train data
        test_size (Optional[float]): Proportion of data for test (0.0-1.0)
        id_column (str): Name of ID column
    
    Returns:
        Tuple of (X_train, X_test, y_train, y_test)
    """
    df = processed_data.data
    features = processed_data.features
    target_col = processed_data.target_column
    
    # Split logic
    if test_condition is not None:
        test_data = df.filter(test_condition)
        if train_condition is not None:
            train_data = df.filter(train_condition)
        else:
            train_data = df.filter(~test_condition)
    elif test_size is not None:
        n_test = int(len(df) * test_size)
        test_data = df.tail(n_test)
        train_data = df.head(-n_test)
    else:
        # Default: last 20% for test
        n_test = int(len(df) * 0.2)
        test_data = df.tail(n_test)
        train_data = df.head(-n_test)
    
    # Extract features
    X_train = train_data.select(features)
    X_test = test_data.select(features)
    
    # Extract targets if available
    y_train = train_data.get_column(target_col) if target_col else None
    y_test = test_data.get_column(target_col) if target_col else None
    
    return X_train, X_test, y_train, y_test

def convert_ret_to_signal(
    ret_arr: np.ndarray,
    params: RetToSignalParameters
) -> np.ndarray:
    """
    Converts raw model predictions (expected returns) into a trading signal.

    Args:
        ret_arr (np.ndarray): The array of predicted returns.
        params (RetToSignalParameters): Parameters for scaling and clipping the signal.

    Returns:
        np.ndarray: The resulting trading signal, clipped between min and max values.
    """
    return np.clip(
        ret_arr * params.signal_multiplier + 1, 
        params.min_signal, 
        params.max_signal
    )

df = pl.read_csv("/kaggle/input/hull-tactical-market-prediction/train.csv")    
processed = preprocess_hull_tactical_data(df, target_column='forward_returns')
    
custom_features = ['S1', 'S2', 'E2', 'P9']
processed = preprocess_hull_tactical_data(
    df, 
    vars_to_keep=custom_features, 
    scale_features=False
)

X_train, X_test, y_train, y_test = split_preprocessed_data(
    processed, 
    test_size=0.2
)
    
print("Function definitions loaded. Ready to preprocess your DataFrames!")

Input data shape: (8990, 98)
Casting numeric columns...
Creating engineered features...
Selecting columns: ['date_id', 'S2', 'E2', 'E3', 'P9', 'S1', 'S5', 'I2', 'P8', 'P10', 'P12', 'P13', 'U1', 'U2']
Handling null values...
Data shape after cleaning: (7479, 14)
Scaling features...
Preprocessing completed successfully!
Final shape: (7479, 14)
Features: ['S2', 'E2', 'E3', 'P9', 'S1', 'S5', 'I2', 'P8', 'P10', 'P12', 'P13', 'U1', 'U2']
Input data shape: (8990, 98)
Casting numeric columns...
Creating engineered features...
Selecting columns: ['date_id', 'forward_returns', 'S2', 'E2', 'E3', 'P9', 'S1', 'S5', 'I2', 'P8', 'P10', 'P12', 'P13', 'U1', 'U2']
Handling null values...
Data shape after cleaning: (7479, 15)
Scaling features...
Preprocessing completed successfully!
Final shape: (7479, 15)
Features: ['S2', 'E2', 'E3', 'P9', 'S1', 'S5', 'I2', 'P8', 'P10', 'P12', 'P13', 'U1', 'U2']
Input data shape: (8990, 98)
Casting numeric columns...
Creating engineered features...
Selecting columns: ['

In [18]:
X_train

S1,S2,E2,P9
f64,f64,f64,f64
-1.059103,1.208733,2.021695,0.652778
-1.064294,1.207146,2.020056,0.653439
-1.063939,1.088541,2.088565,0.654101
-1.063491,0.618246,2.094246,0.654762
-1.068476,0.618119,2.082656,0.655423
…,…,…,…
-0.008385,1.317959,1.355683,0.003968
-0.006109,1.102926,1.321175,0.001323
-0.126787,0.959386,1.447855,0.003307


In [19]:
improved_catboost_params = {'iterations': 3000,
                            'learning_rate': 0.01,
                            'depth': 6,
                            'l2_leaf_reg': 5.0,
                            'min_child_samples': 100,
                            'colsample_bylevel': 0.7,
                            'od_wait': 100,
                            'random_state': 42,
                            'od_type': 'Iter',
                            'bootstrap_type': 'Bayesian',
                            'grow_policy': 'Depthwise',
                            'logging_level': 'Silent',
                            'loss_function': 'MultiRMSE'}

R_Forest_parm = {'n_estimators': 100,
                 'min_samples_split': 5,
                 'max_depth': 15,
                 'min_samples_leaf': 3,
                 'max_features': 'sqrt',
                 'random_state': 42}
        
Extra_parm = {'n_estimators': 100,
              'min_samples_split': 5,
              'max_depth': 12,
              'min_samples_leaf': 3,
              'max_features': 'sqrt',
              'random_state': 42}
        
XGB_R_parm = {"n_estimators": 1500,
              "learning_rate": 0.05, 
              "max_depth": 6,
              "subsample": 0.8, 
              "colsample_bytree": 0.7,
              "reg_alpha": 1.0,
              "reg_lambda": 1.0,
              "random_state": 42}

LGBM_R_parm = {"n_estimators": 1500,
               "learning_rate": 0.05,
               "num_leaves": 50,
               "max_depth": 8,
               "reg_alpha": 1.0,
               "reg_lambda": 1.0,
               "random_state": 42,
               'verbosity': -1}

DecisionTree = {'criterion': 'poisson',
                'max_depth': 6}

GB_parm = {"learning_rate": 0.1,
           "min_samples_split": 500,
           "min_samples_leaf": 50,
           "max_depth": 8,
           "max_features": 'sqrt',
           "subsample": 0.8,
           "random_state": 10}

CatBoost = CatBoostRegressor(**improved_catboost_params)
XGBoost = XGBRegressor(**XGB_R_parm)
LGBM = LGBMRegressor(**LGBM_R_parm)
RandomForest = RandomForestRegressor(**R_Forest_parm)
ExtraTrees = ExtraTreesRegressor(**Extra_parm)
GBRegressor = GradientBoostingRegressor(**GB_parm)
        
DTRegressor = DecisionTreeRegressor(**DecisionTree)


estimators = [('CatBoost', CatBoost), ('XGBoost', XGBoost), ('LGBM', LGBM), ('RandomForest', RandomForest),
              ('ExtraTrees', ExtraTrees), ('GBRegressor', GBRegressor), 
              ('ElasticNet', ElasticNetCV()), ('Lasso', Lasso()), ('RidgeCV', RidgeCV()),
              ('LassoCV', LassoCV()), ('LassoLars', LassoLars())]

model = StackingRegressor(estimators, 
                          final_estimator = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0]), 
                          cv=3)
model.fit(X_train, y_train)

ValueError: y should be a 1d array, got an array of shape () instead.

In [ ]:
def predict(test: pl.DataFrame) -> float:
    test = test.rename({'lagged_forward_returns':'target'})
    df = preprocess_hull_tactical_data(test, target_column='forward_returns')
    raw_pred = model.predict(X_test)[0]
    return raw_pred
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))